In [1]:
import os
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from scipy.interpolate import make_splprep
from scipy.signal import savgol_filter

BLANK = "-"

In [ ]:
def load_language(path: str):
    tokens = [BLANK]
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                tokens.append(line)

    token2idx = {tok: i for i, tok in enumerate(tokens)}
    num_classes = len(tokens)
    return tokens, token2idx, num_classes


def encode_label(processed_label: str, token2idx: dict):
    parts = processed_label.split(BLANK)
    if len(parts)>1:
        parts = parts[1:-1]
    indices = [token2idx[part] for part in parts]
    return indices

# ---------------------------------------------------------------------------
# 2. Feature Extraction
# ---------------------------------------------------------------------------

def extract_features(sample: np.ndarray):
    T = sample.shape[0]
    diff = np.diff(sample, axis=0, prepend=sample[:1, :])  # (N, 2)
    speed = np.linalg.norm(diff, axis=1, keepdims=True)     # (N, 1)
    speed_scaled = speed*10
    diff_norm = diff / (speed + 1e-9)                        # (N, 2)

    # Turning angle between consecutive segment pairs
    angle = np.zeros(T, dtype=np.float32)
    if T >= 3:
        v1 = sample[1:-1] - sample[:-2]   # (T-2, 2)
        v2 = sample[2:]   - sample[1:-1]   # (T-2, 2)
        cross = v1[:, 0] * v2[:, 1] - v1[:, 1] * v2[:, 0]
        dot   = v1[:, 0] * v2[:, 0] + v1[:, 1] * v2[:, 1]
        angle[1:-1] = np.arctan2(cross, dot)
    angle_norm = angle / np.pi


    features = np.concatenate([sample, diff_norm, speed_scaled, angle_norm[:, None]], axis=1)
    return features.astype(np.float32)


# ---------------------------------------------------------------------------
# 3. Dataset
# ---------------------------------------------------------------------------

class AirHandwritingDataset(Dataset):
    def __init__(self, data_dir: str, label_path: str, language_path: str):
        self.tokens, self.token2idx, self.num_classes = load_language(language_path)
        self.word_labels = {}
        self.samples = []
        self.sample_words = []  # track which word each sample belongs to
        with open(label_path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                line = line.strip()
                if i == 0: continue
                parts = line.split(",")
                origin, process = parts
                word = origin.strip()
                label_indices = encode_label(process.strip(), self.token2idx)
                self.word_labels[word] = label_indices

                folder_path = os.path.join(data_dir, word)
                for csv_file in sorted(os.listdir(folder_path)):
                    csv_path = os.path.join(folder_path, csv_file)
                    self.samples.append((csv_path, label_indices))
                    self.sample_words.append(word)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        csv_path, label_indices = self.samples[idx]
        df = pd.read_csv(csv_path)
        coord = df.to_numpy().astype(np.float32)
        features = extract_features(sample=coord)  # (T, 5)
        features = torch.from_numpy(features)
        label = torch.tensor(label_indices, dtype=torch.long)
        return features, label


def collate_fn(batch):
    features_list, labels_list = zip(*batch)

    input_lengths = torch.tensor([f.size(0) for f in features_list], dtype=torch.long)
    label_lengths = torch.tensor([l.size(0) for l in labels_list], dtype=torch.long)

    features_padded = pad_sequence(features_list, batch_first=True, padding_value=0.0)
    labels_concat = torch.cat(labels_list)

    return features_padded, labels_concat, input_lengths, label_lengths


# ---------------------------------------------------------------------------
# 4. Model: Transformer + CTC
# ---------------------------------------------------------------------------

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, :x.size(1)]


class AirHandwritingTransformer(nn.Module):

    def __init__(
        self,
        num_classes: int,
        input_dim: int = 6,
        d_model: int = 128,
        nhead: int = 4,
        num_encoder_layers: int = 4,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
        max_len: int = 1000,
    ):
        super().__init__()
        self.d_model = d_model
        self.input_proj = nn.Linear(input_dim, d_model)

        # Conv1D block: local features + 2x sequence downsampling
        self.conv_block = nn.Sequential(
            nn.Conv1d(d_model, d_model, kernel_size=5, stride=1, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(d_model),
            nn.Dropout(dropout),
            nn.Conv1d(d_model, d_model, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(d_model),
            nn.Dropout(dropout),
            nn.Conv1d(d_model, d_model, kernel_size=5, stride=2, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(d_model),
            nn.Dropout(dropout),
        )

        self.pos_encoder = PositionalEncoding(d_model, max_len, dropout)
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_encoder_layers,
        )

        self.output_proj = nn.Linear(d_model, num_classes)

    def forward(self, src, src_lengths=None):
        x = self.input_proj(src)  # (batch, T, d_model)

        # Conv1D: (batch, T, d_model) -> (batch, d_model, T) -> conv -> (batch, T', d_model)
        x = x.transpose(1, 2)
        x = self.conv_block(x)
        x = x.transpose(1, 2)

        # Recompute lengths after stride-2 conv (kernel=3, padding=1):
        # out_len = floor((in_len - 1) / 2) + 1
        if src_lengths is not None:
            # 2 downsample layer in conv block
            src_lengths = (src_lengths - 1) // 2 + 1
            src_lengths = (src_lengths - 1) // 2 + 1

        # Padding mask for reduced sequence
        src_key_padding_mask = None
        if src_lengths is not None:
            batch_size, max_len, _ = x.size()
            src_key_padding_mask = torch.arange(max_len, device=x.device).unsqueeze(0) >= src_lengths.unsqueeze(1)

        x = self.pos_encoder(x)
        x = self.transformer_encoder(x, src_key_padding_mask=src_key_padding_mask)

        logits = self.output_proj(x)  # (batch, T', num_classes)
        log_probs = F.log_softmax(logits, dim=-1)

        log_probs = log_probs.permute(1, 0, 2)  # (T', batch, num_classes)

        return log_probs, src_lengths


In [ ]:

# ---------------------------------------------------------------------------
# 5. Training
# ---------------------------------------------------------------------------

def train(
    data_dir: str,
    label_path: str,
    language_path: str,
    num_epochs: int = 100,
    batch_size: int = 32,
    lr: float = 1e-3,
    d_model: int = 128,
    nhead: int = 4,
    num_encoder_layers: int = 4,
    dim_feedforward: int = 256,
    dropout: float = 0.1,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
    save_path: str = "model.pth",
):
    dataset = AirHandwritingDataset(data_dir, label_path, language_path)
    tokens = dataset.tokens
    blank_idx = dataset.token2idx[BLANK]
    print(f"Dataset size: {len(dataset)} samples")
    print(f"Vocabulary size: {dataset.num_classes} (including CTC blank)")

    # Stratified split: each word contributes 80% samples to train, 20% to val
    word_to_indices = {}
    for idx, word in enumerate(dataset.sample_words):
        word_to_indices.setdefault(word, []).append(idx)

    train_indices, val_indices = [], []
    for word, indices in word_to_indices.items():
        random.shuffle(indices)
        split = int(0.8 * len(indices))
        train_indices.extend(indices[:split])
        val_indices.extend(indices[split:])

    train_dataset = torch.utils.data.Subset(dataset, train_indices)
    val_dataset = torch.utils.data.Subset(dataset, val_indices)
    print(f"Stratified split: {len(word_to_indices)} words, "
          f"{len(train_indices)} train / {len(val_indices)} val samples")

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        collate_fn=collate_fn, num_workers=2, pin_memory=True,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_fn, num_workers=2, pin_memory=True,
    )

    # Model
    model = AirHandwritingTransformer(
        num_classes=dataset.num_classes,
        input_dim=6,
        d_model=d_model,
        nhead=nhead,
        num_encoder_layers=num_encoder_layers,
        dim_feedforward=dim_feedforward,
        dropout=dropout,
    ).to(device)

    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

    # Optimizer & scheduler with warm-up
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    warmup_epochs = 10
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1e-2, end_factor=1.0, total_iters=warmup_epochs,
    )
    plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=5,
    )

    # CTC loss (blank=0)
    ctc_loss_fn = nn.CTCLoss(blank=0, zero_infinity=True)

    best_val_loss = float("inf")

    for epoch in range(1, num_epochs + 1):
        # --- Training ---
        model.train()
        train_loss = 0.0
        for features, labels, input_lengths, label_lengths in train_loader:
            features = features.to(device)
            labels = labels.to(device)
            input_lengths = input_lengths.to(device)
            label_lengths = label_lengths.to(device)

            log_probs, out_lengths = model(features, input_lengths)  # (T', batch, C)

            loss = ctc_loss_fn(log_probs, labels, out_lengths, label_lengths)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # --- Validation ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for features, labels, input_lengths, label_lengths in val_loader:
                features = features.to(device)
                labels = labels.to(device)
                input_lengths = input_lengths.to(device)
                label_lengths = label_lengths.to(device)

                log_probs, out_lengths = model(features, input_lengths)
                loss = ctc_loss_fn(log_probs, labels, out_lengths, label_lengths)
                val_loss += loss.item()

        val_loss /= len(val_loader)

        # Warm-up for first 10 epochs, then ReduceLROnPlateau
        if epoch <= warmup_epochs:
            warmup_scheduler.step()
        else:
            plateau_scheduler.step(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]
        print(f"Epoch {epoch:03d} | Train Loss: {train_loss:.4f} | "
              f"Val Loss: {val_loss:.4f} | LR: {current_lr:.6f}")

        # Print 3 sample predictions every 5 epochs
        if epoch % 5 == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                # Grab first batch from val_loader
                sample_feats, sample_labels, sample_in_lens, sample_lab_lens = \
                    next(iter(val_loader))
                sample_feats = sample_feats.to(device)
                sample_in_lens = sample_in_lens.to(device)

                sample_log_probs, sample_out_lens = model(sample_feats, sample_in_lens)  # (T', B, C)

                # Split concatenated labels back per sample
                offset = 0
                n_show = min(3, sample_feats.size(0))
                for s in range(n_show):
                    lab_len = sample_lab_lens[s].item()
                    gt_indices = sample_labels[offset:offset + lab_len].tolist()
                    offset += lab_len

                    gt_str = "".join(tokens[idx] for idx in gt_indices)
                    pred_tokens = greedy_decode(
                        sample_log_probs[:sample_out_lens[s], s, :],
                        tokens, blank_idx,
                    )
                    pred_str = "".join(pred_tokens)
                    print(f"  [{s}] GT  : {gt_str}")
                    print(f"  [{s}] Pred: {pred_str}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
            print(f"  -> Saved best model (val_loss={val_loss:.4f})")

    print(f"\nTraining complete. Best val loss: {best_val_loss:.4f}")


In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
DATA_DIR = os.path.join(BASE_DIR, "..", "Vie_air_handwriting_dataset", "processed_data")
LABEL_PATH = os.path.join(BASE_DIR, "processed_label.txt")
LANGUAGE_PATH = os.path.join(BASE_DIR, "language.txt")
MODEL_PATH = os.path.join(BASE_DIR, "best_model.pth")


model = train(
    data_dir=DATA_DIR,
    label_path=LABEL_PATH,
    language_path=LANGUAGE_PATH,
    num_epochs=100,
    batch_size=32,
    lr=1e-3,
    d_model=128,
    nhead=4,
    num_encoder_layers=4,
    dim_feedforward=256,
    dropout=0.1,
    save_path=MODEL_PATH,
)


